# Install Library

In [ ]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.5 MB/s eta 0:00:00


# Import Library

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from google.colab import drive

# Connect to Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path folder tempat file CSV berada
folder_path = "/content/drive/MyDrive/File/CODES/Summarize/3_Summary_Kompas_50_Base1/Sum_Kompas_50_Base1"

# Mengambil semua file dengan ekstensi .csv dalam folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith(".csv")]

# Membaca dan menggabungkan semua file
dataframes = []
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Menggabungkan semua DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
# Menyimpan hasil penggabungan ke file baru
output_path = os.path.join(folder_path, "/content/drive/MyDrive/File/CODES/Similarity/3_Similarity_Kompas_Base1/kompas_sum_50_base1_concat.csv")
combined_df.to_csv(output_path, index=False)

In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8741 entries, 0 to 8740
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         8741 non-null   object
 1   category      8741 non-null   object
 2   publish_date  8741 non-null   object
 3   author        8741 non-null   object
 4   article_url   8741 non-null   object
 5   content       8741 non-null   object
 6   summary       8741 non-null   object
dtypes: object(7)
memory usage: 478.2+ KB


In [ ]:
combined_df.head()

,title,category,publish_date,author,article_url,content,summary
0,Jokowi Pastikan Kerja Sama Indonesia dan Afrik...,Nasional,2024-09-01,"Haryanti Puspa Sari, Ihsanuddin",https://nasional.kompas.com/read/2024/09/01/21...,"Presiden Joko Widodo Jokowi mengatakan, Indone...","Presiden Joko Widodo Jokowi mengatakan, Indone..."
1,Jokowi Perkenalkan Prabowo di Forum Indonesia ...,Nasional,2024-09-01,"Singgih Wiryono, Ihsanuddin",https://nasional.kompas.com/read/2024/09/01/20...,Presiden Joko Widodo Jokowi memperkenalkan pre...,Presiden Joko Widodo Jokowi memperkenalkan pre...
2,Anies Dinilai Punya Modal Politik untuk Bikin ...,Nasional,2024-09-01,"Syakirun Ni'am, Icha Rastika",https://nasional.kompas.com/read/2024/09/01/13...,Pengamat politik Universitas Islam Negeri UIN ...,Pengamat politik Universitas Islam Negeri UIN ...
3,"Prabowo: Kalau Pun Koruptor Lari ke Antartika,...",Nasional,2024-09-01,"Singgih Wiryono, Icha Rastika",https://nasional.kompas.com/read/2024/09/01/13...,Ketua Umum Ketum Partai Gerindra sekaligus pre...,Ketua Umum Ketum Partai Gerindra sekaligus pre...
4,PDI-P Sebenarnya Sudah Dapat Titik Temu dengan...,Nasional,2024-09-01,"Nicholas Ryan Aditya, Icha Rastika",https://nasional.kompas.com/read/2024/09/01/11...,Ketua DPP PDIP Deddy Yevri Sitorus mengungkapk...,Ketua DPP PDIP Deddy Yevri Sitorus mengungkapk...


In [ ]:
nan_summary = combined_df[combined_df['summary'].isna()]


nan_summary

,title,category,publish_date,author,article_url,content,summary


In [ ]:
combined_df=combined_df.dropna()

# Duplicate


In [ ]:
data = combined_df.copy()

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8741 entries, 0 to 8740
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         8741 non-null   object
 1   category      8741 non-null   object
 2   publish_date  8741 non-null   object
 3   author        8741 non-null   object
 4   article_url   8741 non-null   object
 5   content       8741 non-null   object
 6   summary       8741 non-null   object
dtypes: object(7)
memory usage: 478.2+ KB


# Tokenize

Mencoba kode lain agar tokenize lebih mudah di pahami

# Word Embedding

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Memuat IndoBERT dan tokenizer
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1", use_fast=True)
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [ ]:
# Fungsi untuk mendapatkan embedding dari IndoBERT
def get_embeddings(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    # Mengambil hidden states dari layer terakhir dan rata-rata embeddingsnya
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk mengonversi tokenized data ke dalam bentuk input_ids dan mendapatkan embedding
def get_input_ids_and_embeddings(data_column):
    input_ids_list = []
    embeddings_list = []

    for text in data_column:
        # Tokenisasi
        tokens = tokenizer(text, padding='max_length', max_length=256, truncation=True, return_tensors="pt")
        input_ids = tokens['input_ids']  # Ambil input_ids
        input_ids_list.append(input_ids)

        # Mendapatkan embeddings
        embedding = get_embeddings(input_ids)
        embeddings_list.append(embedding)

    return input_ids_list, embeddings_list

# Dapatkan input_ids dan embeddings untuk title dan summary
title_input_ids, title_embeddings = get_input_ids_and_embeddings(data['title'])
summary_input_ids, summary_embeddings = get_input_ids_and_embeddings(data['summary'])


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

## Mengubah embeddings menjadi array numpy

In [ ]:
# Mengubah embeddings menjadi array numpy agar bisa disimpan dalam DataFrame
title_embeddings_array = np.vstack([embedding.numpy() for embedding in title_embeddings])
summary_embeddings_array = np.vstack([embedding.numpy() for embedding in summary_embeddings])

# Membuat DataFrame baru dengan embeddings numerik
data_embeddings = pd.DataFrame({
    'title_embeddings': list(title_embeddings_array),
    'summary_embeddings': list(summary_embeddings_array)
})

# Menyimpan DataFrame embeddings ke dalam file CSV (atau format lain)
# data_embeddings.to_csv('/content/drive/MyDrive/File/CODES/Similarity/Detik Berita/detik_berita_embeddings.csv', index=False)


# Cosine Similarity

In [ ]:
# hitung cosine similarity pada matrix berita
cosine_sim = cosine_similarity(title_embeddings_array, summary_embeddings_array)
print(cosine_sim)

[[0.9121338  0.79156876 0.44168645 ... 0.19062808 0.48020083 0.42453986]
 [0.9077672  0.78760684 0.44080755 ... 0.19101985 0.47997183 0.42366087]
 [0.9092082  0.78886145 0.44180822 ... 0.19118083 0.47919    0.42353004]
 ...
 [0.918085   0.7985732  0.4469295  ... 0.1909951  0.4853959  0.43074894]
 [0.9151063  0.7977096  0.4460653  ... 0.19155326 0.48515156 0.4297856 ]
 [0.92730635 0.8073182  0.45340103 ... 0.18900791 0.48694152 0.43406892]]


In [ ]:
# membuat dataframe dari varible cosine similarity dengan baris dan kolom title
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['summary'], columns=data['title'])

print(cosine_sim_df.shape)

(8741, 8741)


## Check Similarity Result

In [ ]:
# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # The index is a summary, and you need to find the corresponding title
    # to access the similarity value. However, since you have both summaries
    # and titles as indices, you can access the similarity directly using
    # the index and the corresponding title from the 'data' DataFrame.

    title = data.loc[data['summary'] == index, 'title'].iloc[0] # Get the title for the current summary
    similarity_value = cosine_sim_df.loc[index, title]  # Access the similarity using both summary and title

    print(f"Title: {title}")
    print(f"Summary: {index}")
    print(f"Cosine Similarity: {similarity_value}\n")
    print("-" * 50)  # Pemisah antar pasangan

Streaming output truncated to the last 5000 lines.
Title: Jokowi Sebut Kualitas Bibit dan Metode Panen Bakal Tingkatkan Produksi Kelapa RI
Summary: Presiden Joko Widodo menekankan, kualitas bibit dan metode cara panen sangat berpengaruh terhadap peningkatan produksi kelapa di Indonesia. Kepala Negara menuturkan, menyiapkan cara panen bisa dilakukan dengan menyiapkan pemetik buat kelapa. "Kalau kelapanya 20 meter, berapa juta pohon kelapa, berarti berapa orang yang harus disiapkan untuk memetik kelapa itu," ucap dia. Selain dua hal itu, pemeliharaan juga perlu dilakukan. Buahbuah kelapa yang ditanam harus dipelihara agar hasilnya pun bagus. " Biasanya kita menanam, dibiarkan, berbuah, baru diambil. Jadi kualitas bibit, pemeliharaan, dan metode cara panen itu penting," kata Jokowi. Adapun saat ini, Indonesia memiliki luas lahan kelapa sebesar 3,8 juta hektare dengan produksi 2,8 juta ton per tahun. Ini sebuah angka yang sangat besar dan bisa ditingkatkan lagi kalau kita serius.
Cosine Si

In [ ]:
# List to store the results
results = []

# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # Mendapatkan judul yang sesuai dengan summary berdasarkan index
    title = data.loc[data['summary'] == index, 'title'].iloc[0]
    similarity_value = cosine_sim_df.loc[index, title]  # Mengakses nilai similarity

    # Menyimpan pasangan dan nilai similarity ke dalam list
    results.append([title, index, similarity_value])

# Membuat DataFrame untuk hasilnya
similarity_df = pd.DataFrame(results, columns=['Title', 'Summary', 'Cosine Similarity'])


In [ ]:
similarity_df.head()

,Title,Summary,Cosine Similarity
0,Jokowi Pastikan Kerja Sama Indonesia dan Afrik...,"Presiden Joko Widodo Jokowi mengatakan, Indone...",0.912134
1,Jokowi Perkenalkan Prabowo di Forum Indonesia ...,Presiden Joko Widodo Jokowi memperkenalkan pre...,0.787607
2,Anies Dinilai Punya Modal Politik untuk Bikin ...,Pengamat politik Universitas Islam Negeri UIN ...,0.441808
3,"Prabowo: Kalau Pun Koruptor Lari ke Antartika,...",Ketua Umum Ketum Partai Gerindra sekaligus pre...,0.915256
4,PDI-P Sebenarnya Sudah Dapat Titik Temu dengan...,Ketua DPP PDIP Deddy Yevri Sitorus mengungkapk...,0.159616


## Save Similarity

In [ ]:
# Menyimpan DataFrame ke file CSV di lokasi yang ditentukan
similarity_df.to_csv('/content/drive/MyDrive/File/CODES/Similarity/3_Similarity_Kompas_Base1/similarity_kompas_base1.csv', index=False)